# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

def resolve_data_path(filename):
    path = Path(filename)
    if path.exists():
        return path
    for base in [Path.cwd(), *Path.cwd().parents]:
        matches = list(base.rglob(filename))
        if matches:
            return matches[0]
    return path

df = pd.read_csv(resolve_data_path("customers.csv"))
df.head()


,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No


In [2]:
def load_and_clean_customers(path="customers.csv"):
    data_path = resolve_data_path(path)
    df = pd.read_csv(data_path)
    df["income"] = pd.to_numeric(df["income"], errors="coerce")
    df.loc[df["income"] < 0, "income"] = np.nan
    df["region"] = df["region"].str.strip().str.title()
    df["income"] = df["income"].fillna(df["income"].median())
    return df


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [3]:
df = load_and_clean_customers()
# treat impossible income value as missing before filling
# standardize the region label to consistent capitalization
# median is more robust than mean because income is skewed and includes a bad value
df.isna().sum()


customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64

**Explain each issue:** what you found and how you handled it.

*Your answer:*
I found three data-quality problems before analysis: the `income` column had two missing values, so I filled those with the median income because the income distribution is skewed and the median is more robust than the mean; the region label `east` was lowercase, so I standardized it to `East` with `.str.title()`; and the income value `-50` was impossible for a salary, so I treated it as missing and then filled it using the same median. After cleaning, there were no missing values left.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [4]:
if "df" not in globals():
    df = load_and_clean_customers()

r, p = stats.pearsonr(df["income"], df["spending"])
print(f"correlation = {r:.3f}, p-value = {p:.6f}")
r, p


correlation = 0.717, p-value = 0.000000


(np.float64(0.7172222861355748), np.float64(1.9185294399175956e-07))

*Your answer:*
Yes - customers with higher income tend to spend more, with a positive Pearson correlation of r = 0.717 (p < 0.001).


### Q2. Is a customer's region related to whether they subscribed?

In [ ]:
if "df" not in globals():
    df = load_and_clean_customers()

region_subscribed = pd.crosstab(df["region"], df["subscribed"])
chi2, p, dof, expected = stats.chi2_contingency(region_subscribed)
print(region_subscribed)
print(f"chi-square = {chi2:.3f}, p-value = {p:.4f}")
print(expected)


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
chi-square = 13.162, p-value = 0.0014
[[10.5 10.5]
 [ 4.5  4.5]
 [ 5.   5. ]]


: 

*Your answer:*
Yes - region and subscription are related, chi-square statistic = 13.16, p = 0.0014, so the distribution of subscriptions differs by region.


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [ ]:
if "df" not in globals():
    df = load_and_clean_customers()
if "sm" not in globals():
    import statsmodels.api as sm

X = sm.add_constant(df[["income", "age"]])
model = sm.OLS(df["spending"], X).fit()
print(model.summary())
print(f"R-squared = {model.rsquared:.3f}")
print(f"p-value for income = {model.pvalues['income']:.4f}")
print(f"p-value for age = {model.pvalues['age']:.4f}")


*Your answer:*
Spending can be predicted from income and age together using multiple regression; the model explains about 52.0% of the variation in spending (R-squared = 0.520), and income is a significant predictor (p = 0.003) while age is not (p = 0.515).


### Q4. Which region has the most customers, and what is the average income overall?

In [ ]:
if "df" not in globals():
    df = load_and_clean_customers()

region_counts = df["region"].value_counts()
avg_income = df["income"].mean()
print(region_counts)
print(f"Average income overall = {avg_income:.2f}")


*Your answer:*
The East region has the most customers (21), and the overall average income is $86.46k.


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
The choice was determined by the data types and the question structure: two continuous variables called for correlation, two categorical variables called for a chi-square test, one continuous outcome with two numeric predictors called for multiple regression, and a count-plus-average question called for descriptive statistics.

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*
No - correlation does not prove causation, because higher income may be associated with higher spending due to other factors such as age, preferences, or lifestyle; this analysis only shows that the two variables move together.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(pd.crosstab(df['region'], df['subscribed']))
print(expected)
print(expected >= 5)


*Your answer:*
No - not all expected counts meet the chi-square assumption, because the expected counts include 4.5 for the South region in both subscription categories, which is below 5; I would tell the company that the chi-square result is suggestive but only moderately trustworthy because one assumption is violated.


### Push your code to GitHub